# SwSL Recognition — Full Reviewer-Response Training Pipeline

This notebook retrains the Swahili Sign Language system from the raw videos and produces
**every number, table and figure** needed to answer Reviewer A and Reviewer B.

**Before you run anything, read these three notes.**

1. **Your dataset has 8 class folders** (`baba, habari, hedhi, kula, mama, nenda, njema, siku`)
   but the manuscript reports **7 classes**. `siku` is the extra one. Set `CLASSES` in the
   config cell to whichever set you intend to publish. Every table in the paper must match
   that choice. The notebook auto-detects folders and will tell you what it found.
2. **Signer identity is parsed from the filename prefix** (`Britney_002.mp4` -> signer
   `Britney`). This is what makes the signer-independent evaluation in Section 12 possible,
   which is Reviewer B's comment 15. If your filenames use a different convention, fix
   `parse_signer()`.
3. **The split happens before augmentation, and augmentation touches only the training
   partition.** This is Reviewer A comment 6 and it is enforced structurally, not by
   convention.

---

### Which cell answers which reviewer comment

| Section | Reviewer comment | What it produces |
|---|---|---|
| 3. Dataset inventory | A6, B4 | Per-class / per-signer counts, clip durations |
| 4. Landmark extraction | B5, B6 | Cached landmarks, detection-quality log |
| 5. Normalization | **B6** | Explicit, leak-free normalization with documented scope |
| 6. Splitting | **A6**, B15 | Exact train/val/test counts, before and after augmentation |
| 7. Augmentation | **A6** | Train-only augmentation, 7 variants |
| 8. Models | A7 | Proposed + 7 baselines, one protocol |
| 9-10. Training | **A7**, B10 | Baseline table, mean +/- std over seeds, true parameter counts |
| 11. Evaluation | **B13**, A9 | Confusion matrix as `17 (94.4%)`, per-class metrics, McNemar test |
| 12. Ablations | A4, A5, **A10**, **B16** | 10 variants including a `with_face` test |
| 13. Attention | **A5** | Attention weights and saliency figure |
| 14. Gesture characterization | **B4** | Handedness, articulation height, movement magnitude per class |
| 15. Signer-independent | **B15** | Leave-one-signer-out results |
| 16. Parameter audit | **B10** | Layer-by-layer parameter table, resolves the 177,607 discrepancy |
| 17. Runtime benchmark | **B14** | Extraction FPS, inference latency, end-to-end delay |
| 18. Speech synthesis | **B7** | MMS-VITS checkpoint, config, generated audio |
| 19. Diagrams | **A15**, **B5** | System architecture and landmark pipeline figures |
| 20. Export | A14 | Zipped `outputs/` with every CSV and PNG |


## 1. Environment

In [ ]:
# Colab: Runtime > Change runtime type > GPU (T4) is recommended but not required.
!pip -q install mediapipe==0.10.14 2>/dev/null
!pip -q install transformers==4.44.2 2>/dev/null
!pip -q install statsmodels 2>/dev/null

import os, sys, json, math, time, glob, random, shutil, warnings, itertools
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200,
                            'savefig.bbox': 'tight', 'font.size': 10})

print('TensorFlow :', tf.__version__)
print('OpenCV     :', cv2.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU') or 'none (CPU only)')


## 2. Configuration

Everything that affects a reported number is set here and written to `outputs/config.json`,
so the manuscript can quote the exact configuration used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------- PATHS ----
# Point this at the folder that CONTAINS the class subfolders (baba/, habari/, ...).
RAW_VIDEO_DIR = '/content/drive/MyDrive/raw_videos'   # <-- EDIT THIS
OUT_DIR       = '/content/outputs'
CACHE_DIR     = '/content/landmark_cache'

# --------------------------------------------------------------- CLASSES ---
# None = auto-detect every subfolder. To publish the 7-class version used in the
# manuscript, list them explicitly and leave 'siku' out:
#   CLASSES = ['baba','habari','hedhi','kula','mama','nenda','njema']
CLASSES = None

# ------------------------------------------------------------- EXTRACTION --
SEQ_LEN            = 60      # standardized frames per clip
INCLUDE_FACE       = True    # extract face landmarks too, so the with_face ablation (A10) can run
MIN_DET_CONF       = 0.5     # MediaPipe min_detection_confidence
MIN_TRK_CONF       = 0.5     # MediaPipe min_tracking_confidence
QUALITY_THRESHOLD  = 0.90    # min fraction of frames with a pose detection to keep a clip
LENGTH_MODE        = 'pad_truncate'   # 'pad_truncate' (as published) or 'resample'

# ---------------------------------------------------------- NORMALIZATION --
# Answers Reviewer B comment 6. The scope is explicit and there is no leakage:
#   'train_stats'   : min/max per coordinate dimension computed on the TRAINING split only
#   'per_sequence'  : min/max within each clip
#   'per_frame'     : min/max within each frame  (what the current text implies)
#   'body_anchored' : translate to mid-shoulder origin, scale by shoulder width (recommended)
NORMALIZATION = 'train_stats'

# ------------------------------------------------------------- SPLITTING ---
SPLIT_SEED   = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# ---------------------------------------------------------- AUGMENTATION ---
# Applied to the TRAINING partition only, after the split (Reviewer A comment 6).
AUG_STRETCH = [0.9, 1.1]
AUG_NOISE   = [0.01, 0.02]
AUG_SHIFT   = [5, -5]

# ---------------------------------------------------------------- TRAINING -
SEEDS        = [1, 2, 3]     # independent initializations -> mean +/- std (Reviewer A comment 7)
EPOCHS       = 200
BATCH_SIZE   = 16
LR           = 5e-4
CLIPNORM     = 1.0
L2_REG       = 1e-3
REC_DROPOUT  = 0.2
DENSE_DROP   = [0.4, 0.3]
ES_PATIENCE  = 30
RLR_PATIENCE = 10

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

POSE_DIM, HAND_DIM, FACE_DIM = 33*4, 21*3, 468*3
FEAT_DIM = POSE_DIM + 2*HAND_DIM                 # 258
FULL_DIM = FEAT_DIM + FACE_DIM if INCLUDE_FACE else FEAT_DIM

print(f'Feature dim (pose+hands) : {FEAT_DIM}')
print(f'Stored dim (incl. face)  : {FULL_DIM}')


## 3. Dataset inventory

Produces the per-class / per-signer composition table (manuscript Table 2) and the clip
duration statistics (Table 3). Run this first: it tells you how many classes and signers
you actually have before anything expensive starts.

In [ ]:
import re

def parse_signer(filename):
    # 'Britney_002.mp4' -> 'Britney'. Adjust if your naming differs.
    stem = Path(filename).stem
    m = re.match(r'^([A-Za-z]+)', stem)
    return m.group(1) if m else 'unknown'

root = Path(RAW_VIDEO_DIR)
assert root.exists(), f'RAW_VIDEO_DIR not found: {root}'

detected = sorted([d.name for d in root.iterdir() if d.is_dir()])
print('Class folders found:', detected)

if CLASSES is None:
    CLASSES = detected
    print(f'\nUsing all {len(CLASSES)} detected folders.')
else:
    missing = [c for c in CLASSES if c not in detected]
    assert not missing, f'Configured classes missing on disk: {missing}'
    print(f'\nUsing {len(CLASSES)} configured classes; ignoring '
          f'{[d for d in detected if d not in CLASSES]}')

VIDEO_EXT = ('.mp4', '.mov', '.MP4', '.MOV', '.avi', '.mkv')
records = []
for cls in CLASSES:
    for f in sorted((root/cls).iterdir()):
        if f.suffix in VIDEO_EXT:
            records.append({'path': str(f), 'label': cls, 'signer': parse_signer(f.name),
                            'filename': f.name})

inv = pd.DataFrame(records)
assert len(inv), 'No videos found. Check RAW_VIDEO_DIR and file extensions.'

print(f'\nTotal clips: {len(inv)}   Signers: {sorted(inv.signer.unique())}')

pivot = inv.pivot_table(index='label', columns='signer', values='filename',
                        aggfunc='count', fill_value=0)
pivot['Total'] = pivot.sum(axis=1)
pivot.loc['Total'] = pivot.sum(axis=0)
display(pivot)
pivot.to_csv(f'{OUT_DIR}/table_dataset_inventory.csv')

# ---- clip duration / frame-count statistics (manuscript Table 3) ----------
stats = []
for r in records:
    cap = cv2.VideoCapture(r['path'])
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    stats.append({'label': r['label'], 'signer': r['signer'], 'frames': n,
                  'fps': round(fps, 2), 'seconds': round(n/fps, 2) if fps else np.nan,
                  'width': w, 'height': h})
clip_stats = pd.DataFrame(stats)
inv = inv.merge(clip_stats[['label','signer','frames','fps','seconds']],
                left_index=True, right_index=True, suffixes=('','_y'))
inv = inv.loc[:, ~inv.columns.str.endswith('_y')]

summary = pd.DataFrame({
    'Attribute': ['Clips', 'Classes', 'Signers', 'Frame rate (fps)',
                  'Frames per clip (min-max)', 'Frames per clip (mean +/- std)',
                  'Duration s (mean +/- std)', 'Resolution(s)'],
    'Value': [len(inv), len(CLASSES), inv.signer.nunique(),
              ', '.join(map(str, sorted(clip_stats.fps.unique()))),
              f'{clip_stats.frames.min()}-{clip_stats.frames.max()}',
              f'{clip_stats.frames.mean():.1f} +/- {clip_stats.frames.std():.1f}',
              f'{clip_stats.seconds.mean():.2f} +/- {clip_stats.seconds.std():.2f}',
              ', '.join(sorted({f'{w}x{h}' for w,h in zip(clip_stats.width, clip_stats.height)}))]})
display(summary)
summary.to_csv(f'{OUT_DIR}/table_dataset_properties.csv', index=False)
inv.to_csv(f'{OUT_DIR}/inventory.csv', index=False)


## 4. Landmark extraction

Runs MediaPipe Holistic over every clip once and caches the result as `.npy`, so all later
experiments reuse the same landmarks. Also records a per-clip detection-quality log, which
is the evidence behind the 90% quality threshold described in the manuscript.

This is the slow cell (roughly 1-3 s per clip). It is safe to interrupt and re-run: cached
clips are skipped.

In [ ]:
import mediapipe as mp
mp_holistic = mp.solutions.holistic

def extract_landmarks(video_path, holistic):
    cap = cv2.VideoCapture(video_path)
    frames, pose_hits, lh_hits, rh_hits = [], 0, 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        res = holistic.process(image)

        if res.pose_landmarks:
            pose = np.array([[p.x, p.y, p.z, p.visibility]
                             for p in res.pose_landmarks.landmark]).flatten()
            pose_hits += 1
        else:
            pose = np.zeros(POSE_DIM)

        if res.left_hand_landmarks:
            lh = np.array([[p.x, p.y, p.z]
                           for p in res.left_hand_landmarks.landmark]).flatten()
            lh_hits += 1
        else:
            lh = np.zeros(HAND_DIM)

        if res.right_hand_landmarks:
            rh = np.array([[p.x, p.y, p.z]
                           for p in res.right_hand_landmarks.landmark]).flatten()
            rh_hits += 1
        else:
            rh = np.zeros(HAND_DIM)

        vec = [pose, lh, rh]
        if INCLUDE_FACE:
            if res.face_landmarks:
                fc = np.array([[p.x, p.y, p.z]
                               for p in res.face_landmarks.landmark]).flatten()
            else:
                fc = np.zeros(FACE_DIM)
            vec.append(fc)

        frames.append(np.concatenate(vec))
    cap.release()

    n = max(len(frames), 1)
    quality = {'n_frames': len(frames),
               'pose_rate': pose_hits/n, 'lh_rate': lh_hits/n, 'rh_rate': rh_hits/n,
               'any_hand_rate': max(lh_hits, rh_hits)/n}
    return (np.array(frames, dtype=np.float32) if frames
            else np.zeros((0, FULL_DIM), np.float32)), quality


qlog, t0 = [], time.time()
with mp_holistic.Holistic(static_image_mode=False, model_complexity=1,
                          refine_face_landmarks=False,
                          min_detection_confidence=MIN_DET_CONF,
                          min_tracking_confidence=MIN_TRK_CONF) as holistic:
    for i, r in inv.iterrows():
        cache_f = Path(CACHE_DIR)/f"{r['label']}__{Path(r['filename']).stem}.npy"
        meta_f  = cache_f.with_suffix('.json')
        if cache_f.exists() and meta_f.exists():
            q = json.load(open(meta_f))
        else:
            seq, q = extract_landmarks(r['path'], holistic)
            np.save(cache_f, seq)
            json.dump(q, open(meta_f, 'w'))
        q.update({'label': r['label'], 'signer': r['signer'],
                  'filename': r['filename'], 'cache': str(cache_f)})
        qlog.append(q)
        if (i+1) % 25 == 0:
            print(f'  {i+1}/{len(inv)} clips  ({time.time()-t0:.0f}s)')

quality_df = pd.DataFrame(qlog)
quality_df.to_csv(f'{OUT_DIR}/landmark_quality_log.csv', index=False)

print(f'\nExtraction finished in {time.time()-t0:.0f}s')
print(f"Mean pose detection rate     : {quality_df.pose_rate.mean():.3f}")
print(f"Mean 'at least one hand' rate: {quality_df.any_hand_rate.mean():.3f}")

below = quality_df[quality_df.pose_rate < QUALITY_THRESHOLD]
print(f'\nClips below the {QUALITY_THRESHOLD:.0%} pose-detection threshold: {len(below)}')
if len(below):
    display(below[['label','signer','filename','pose_rate','any_hand_rate']])
    print('Re-record or exclude these; the manuscript states that sub-threshold clips '
          'were re-recorded rather than silently dropped.')

kept = quality_df[quality_df.pose_rate >= QUALITY_THRESHOLD].reset_index(drop=True)
print(f'\nClips retained: {len(kept)} / {len(quality_df)}')


## 5. Sequence standardization and normalization

**Reviewer B comment 6.** The normalization scope is now explicit. The default
(`train_stats`) computes the min and max of each coordinate dimension *on the training
split only* and applies them everywhere, which is both precisely defined and free of
leakage. Three alternatives are implemented; whichever you choose is written to
`outputs/config.json` so the manuscript can state it exactly.

In [ ]:
def standardize_length(seq, T=SEQ_LEN, mode=LENGTH_MODE):
    if len(seq) == 0:
        return np.zeros((T, seq.shape[1] if seq.ndim > 1 else FULL_DIM), np.float32)
    if mode == 'resample':
        idx = np.linspace(0, len(seq)-1, T)
        out = np.stack([seq[int(round(i))] for i in idx])
        return out.astype(np.float32)
    if len(seq) >= T:
        return seq[:T].astype(np.float32)
    pad = np.zeros((T-len(seq), seq.shape[1]), np.float32)
    return np.concatenate([seq, pad]).astype(np.float32)


def body_anchor(seq):
    # Translate to mid-shoulder origin and scale by shoulder width, per frame.
    out = seq.copy()
    L_SH, R_SH = 11, 12                       # MediaPipe pose indices
    for t in range(len(out)):
        px = out[t, :POSE_DIM].reshape(33, 4)
        if np.allclose(px[:, :3], 0):
            continue
        mid   = (px[L_SH, :3] + px[R_SH, :3]) / 2.0
        width = np.linalg.norm(px[L_SH, :2] - px[R_SH, :2]) or 1.0
        px[:, :3] = (px[:, :3] - mid) / width
        out[t, :POSE_DIM] = px.reshape(-1)
        off = POSE_DIM
        n_hand_blocks = 2 + (1 if INCLUDE_FACE else 0)
        sizes = [HAND_DIM, HAND_DIM] + ([FACE_DIM] if INCLUDE_FACE else [])
        for size in sizes:
            blk = out[t, off:off+size].reshape(-1, 3)
            if not np.allclose(blk, 0):
                blk[:] = (blk - mid) / width
                out[t, off:off+size] = blk.reshape(-1)
            off += size
    return out


# ---- load every retained clip, standardized but not yet normalized --------
X_raw, y_raw, g_raw, f_raw = [], [], [], []
for _, r in kept.iterrows():
    seq = np.load(r['cache'])
    X_raw.append(standardize_length(seq))
    y_raw.append(r['label']); g_raw.append(r['signer']); f_raw.append(r['filename'])

X_raw = np.stack(X_raw)
y_raw = np.array(y_raw); g_raw = np.array(g_raw); f_raw = np.array(f_raw)
print('Loaded tensor:', X_raw.shape, '(clips, frames, features incl. face)' )

CLASS_LIST = sorted(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_LIST)}
y_idx = np.array([CLASS_TO_IDX[c] for c in y_raw])
print('Classes:', CLASS_LIST)


### Normalizers (fitted on training data only)

In [ ]:
class Normalizer:
    def __init__(self, mode=NORMALIZATION):
        self.mode = mode
        self.min_ = None
        self.max_ = None

    def fit(self, X_train):
        if self.mode == 'train_stats':
            flat = X_train.reshape(-1, X_train.shape[-1])
            mask = ~np.all(flat == 0, axis=1)          # ignore padded/undetected frames
            flat = flat[mask] if mask.any() else flat
            self.min_ = flat.min(axis=0)
            self.max_ = flat.max(axis=0)
            rng = self.max_ - self.min_
            rng[rng == 0] = 1.0
            self.range_ = rng
        return self

    def transform(self, X):
        if self.mode == 'train_stats':
            return ((X - self.min_) / self.range_).astype(np.float32)
        if self.mode == 'per_sequence':
            mn = X.min(axis=1, keepdims=True); mx = X.max(axis=1, keepdims=True)
            rng = mx - mn; rng[rng == 0] = 1.0
            return ((X - mn) / rng).astype(np.float32)
        if self.mode == 'per_frame':
            mn = X.min(axis=2, keepdims=True); mx = X.max(axis=2, keepdims=True)
            rng = mx - mn; rng[rng == 0] = 1.0
            return ((X - mn) / rng).astype(np.float32)
        if self.mode == 'body_anchored':
            return np.stack([body_anchor(s) for s in X]).astype(np.float32)
        raise ValueError(self.mode)

print(f"Normalization mode: '{NORMALIZATION}'")
print({'train_stats':  'min/max per dimension from the TRAINING split only (leak-free)',
       'per_sequence': 'min/max within each clip',
       'per_frame':    'min/max within each frame',
       'body_anchored':'mid-shoulder origin, shoulder-width scale'}[NORMALIZATION])


## 6. Splitting — **Reviewer A comment 6**

The split is performed on the **original clips**, before any augmentation. Augmentation is
applied to the training partition only, in the next cell. This makes the arithmetic the
reviewer questioned reproducible and removes any possibility of an augmented sibling of a
test clip appearing in training.

The cell prints the exact original and augmented counts for train / validation / test,
which is what the reviewer asked to be stated explicitly.

In [ ]:
from sklearn.model_selection import train_test_split

def stratified_split(y, seed=SPLIT_SEED):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=(VAL_FRAC+TEST_FRAC),
                               stratify=y, random_state=seed)
    rel = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
    va, te = train_test_split(tmp, test_size=rel, stratify=y[tmp], random_state=seed)
    return tr, va, te

tr_i, va_i, te_i = stratified_split(y_idx)

split_tbl = pd.DataFrame({
    'Partition': ['Train', 'Validation', 'Test', 'Total'],
    'Original clips': [len(tr_i), len(va_i), len(te_i), len(y_idx)],
    'Fraction': [f'{len(tr_i)/len(y_idx):.1%}', f'{len(va_i)/len(y_idx):.1%}',
                 f'{len(te_i)/len(y_idx):.1%}', '100%'],
    'Clips per class': [len(tr_i)//len(CLASS_LIST), len(va_i)//len(CLASS_LIST),
                        len(te_i)//len(CLASS_LIST), len(y_idx)//len(CLASS_LIST)],
})
print('Split performed on ORIGINAL clips, before augmentation:')
display(split_tbl)

print('\nPer-class counts per partition:')
per_class = pd.DataFrame({
    'Train': Counter(y_idx[tr_i]), 'Validation': Counter(y_idx[va_i]),
    'Test': Counter(y_idx[te_i])}).fillna(0).astype(int)
per_class.index = [CLASS_LIST[i] for i in per_class.index]
display(per_class)

print('\nSigner distribution per partition (this is why the protocol is signer-DEPENDENT):')
sig_tbl = pd.DataFrame({
    'Train': Counter(g_raw[tr_i]), 'Validation': Counter(g_raw[va_i]),
    'Test': Counter(g_raw[te_i])}).fillna(0).astype(int)
display(sig_tbl)
sig_tbl.to_csv(f'{OUT_DIR}/table_signer_distribution.csv')


## 7. Augmentation — training partition only

Seven variants per training clip: the original, two temporal stretches, two noise levels and
two temporal shifts. Validation and test partitions are left untouched.

In [ ]:
def temporal_stretch(seq, alpha, T=SEQ_LEN):
    src = np.linspace(0, len(seq)-1, max(int(round(len(seq)*alpha)), 2))
    stretched = np.stack([seq[int(round(i))] for i in src])
    return standardize_length(stretched, T, 'pad_truncate')

def add_noise(seq, sigma, rng):
    out = seq.copy()
    live = ~np.all(out == 0, axis=1)
    out[live] += rng.normal(0, sigma, out[live].shape).astype(np.float32)
    return out

def temporal_shift(seq, k):
    out = np.zeros_like(seq)
    if k >= 0: out[k:] = seq[:len(seq)-k]
    else:      out[:len(seq)+k] = seq[-k:]
    return out

def augment_training_set(X, y, g, seed=SPLIT_SEED):
    rng = np.random.default_rng(seed)
    Xa, ya, ga, tag = [], [], [], []
    for xi, yi, gi in zip(X, y, g):
        variants = [('original', xi)]
        for a in AUG_STRETCH: variants.append((f'stretch_{a}', temporal_stretch(xi, a)))
        for s in AUG_NOISE:   variants.append((f'noise_{s}',   add_noise(xi, s, rng)))
        for k in AUG_SHIFT:   variants.append((f'shift_{k:+d}', temporal_shift(xi, k)))
        for name, v in variants:
            Xa.append(v); ya.append(yi); ga.append(gi); tag.append(name)
    return np.stack(Xa), np.array(ya), np.array(ga), np.array(tag)

X_tr_aug, y_tr_aug, g_tr_aug, aug_tag = augment_training_set(
    X_raw[tr_i], y_idx[tr_i], g_raw[tr_i])

aug_tbl = pd.DataFrame({
    'Partition': ['Train', 'Validation', 'Test', 'Total'],
    'Original clips': [len(tr_i), len(va_i), len(te_i), len(y_idx)],
    'After augmentation': [len(X_tr_aug), len(va_i), len(te_i),
                           len(X_tr_aug)+len(va_i)+len(te_i)],
    'Augmentation factor': [f'{len(X_tr_aug)//max(len(tr_i),1)}x', '1x (none)',
                            '1x (none)', '-'],
})
print('THIS IS THE TABLE THAT ANSWERS REVIEWER A COMMENT 6:')
display(aug_tbl)
aug_tbl.to_csv(f'{OUT_DIR}/table_split_and_augmentation.csv', index=False)

print('\nVariants per training clip:', sorted(set(aug_tag)))
print(f'Test set = {len(te_i)} original clips, '
      f'{len(te_i)//len(CLASS_LIST)} per class across {len(CLASS_LIST)} classes.')


## 8. Feature views and normalization fit

`FEAT_DIM` (258) is the published configuration. The `pose_only`, `hands_only` and
`with_face` views exist so the ablations in Section 12 can be run on identical splits.

In [ ]:
SLICES = {
    'full_258':   slice(0, FEAT_DIM),
    'pose_only':  slice(0, POSE_DIM),
    'hands_only': slice(POSE_DIM, FEAT_DIM),
}
if INCLUDE_FACE:
    SLICES['with_face'] = slice(0, FULL_DIM)

def build_view(view='full_258'):
    sl = SLICES[view]
    norm = Normalizer().fit(X_tr_aug[:, :, sl])
    return {
        'Xtr': norm.transform(X_tr_aug[:, :, sl]), 'ytr': y_tr_aug,
        'Xva': norm.transform(X_raw[va_i][:, :, sl]), 'yva': y_idx[va_i],
        'Xte': norm.transform(X_raw[te_i][:, :, sl]), 'yte': y_idx[te_i],
        'dim': (sl.stop - sl.start), 'norm': norm,
    }

DATA = build_view('full_258')
print({k: (v.shape if hasattr(v, 'shape') else v)
       for k, v in DATA.items() if k in ('Xtr','ytr','Xva','yva','Xte','yte','dim')})

json.dump({'classes': CLASS_LIST, 'seq_len': SEQ_LEN, 'feat_dim': FEAT_DIM,
           'include_face': INCLUDE_FACE, 'normalization': NORMALIZATION,
           'length_mode': LENGTH_MODE, 'split_seed': SPLIT_SEED,
           'fractions': [TRAIN_FRAC, VAL_FRAC, TEST_FRAC],
           'augmentation': {'stretch': AUG_STRETCH, 'noise': AUG_NOISE, 'shift': AUG_SHIFT},
           'n_train_original': int(len(tr_i)), 'n_train_augmented': int(len(X_tr_aug)),
           'n_val': int(len(va_i)), 'n_test': int(len(te_i)),
           'mediapipe': {'min_detection_confidence': MIN_DET_CONF,
                         'min_tracking_confidence': MIN_TRK_CONF,
                         'model_complexity': 1},
           'training': {'optimizer': 'adam', 'lr': LR, 'clipnorm': CLIPNORM,
                        'l2': L2_REG, 'recurrent_dropout': REC_DROPOUT,
                        'dense_dropout': DENSE_DROP, 'batch_size': BATCH_SIZE,
                        'epochs': EPOCHS, 'seeds': SEEDS}},
          open(f'{OUT_DIR}/config.json', 'w'), indent=2)
print('\nWrote outputs/config.json')


## 9. Model definitions — **Reviewer A comment 7**

The proposed model plus seven baselines spanning the families the reviewer named: MLP,
temporal CNN, CNN-LSTM hybrid, unidirectional LSTM, BiGRU, BiLSTM without attention,
Transformer encoder, and a lightweight spatio-temporal graph network over the landmark
skeleton. All of them consume the identical tensors produced above.

The `TemporalAttention` layer below is the exact mechanism described in Section 3.5 of the
manuscript and is what **Reviewer A comment 5** asks to be made reproducible: additive
scoring, softmax over the 60 time steps, masked padding, weighted sum to a single context
vector.

In [ ]:
from tensorflow.keras import regularizers

class TemporalAttention(layers.Layer):
    # Additive (Bahdanau-style) attention over time.
    #   scores  e_t = v^T tanh(W h_t + b)          -> (batch, T)
    #   weights a_t = softmax(e_t) over T, padding masked
    #   context c   = sum_t a_t h_t                -> (batch, d)
    def __init__(self, **kw):
        super().__init__(**kw)
        self.supports_masking = True

    def build(self, input_shape):
        d = int(input_shape[-1])
        self.W = self.add_weight(shape=(d, d), name='att_W',
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(shape=(d,), name='att_b',
                                 initializer='zeros', trainable=True)
        self.v = self.add_weight(shape=(d, 1), name='att_v',
                                 initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, h, mask=None, return_attention=False):
        e = tf.tanh(tf.tensordot(h, self.W, axes=1) + self.b)      # (B,T,d)
        e = tf.squeeze(tf.tensordot(e, self.v, axes=1), -1)        # (B,T)
        if mask is not None:
            e += (1.0 - tf.cast(mask, e.dtype)) * -1e9
        a = tf.nn.softmax(e, axis=1)                               # (B,T)
        c = tf.reduce_sum(h * tf.expand_dims(a, -1), axis=1)       # (B,d)
        if return_attention:
            return c, a
        return c

    def compute_mask(self, inputs, mask=None):
        return None


def _in(dim):
    return keras.Input(shape=(SEQ_LEN, dim))

def _head(x, n_cls, l2=L2_REG, drops=DENSE_DROP):
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(drops[0])(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(drops[1])(x)
    return layers.Dense(n_cls, activation='softmax')(x)


# ----------------------------------------------------- PROPOSED MODEL -----
def build_proposed(dim, n_cls, bidirectional=True, n_layers=2,
                   pooling='attention', use_bn=True, units=(128, 64)):
    inp = _in(dim)
    x = layers.Masking(mask_value=0.0)(inp)
    RNN = (lambda u, rs: layers.Bidirectional(
              layers.LSTM(u, return_sequences=rs, recurrent_dropout=REC_DROPOUT,
                          kernel_regularizer=regularizers.l2(L2_REG)))
           ) if bidirectional else (
           lambda u, rs: layers.LSTM(u, return_sequences=rs, recurrent_dropout=REC_DROPOUT,
                                     kernel_regularizer=regularizers.l2(L2_REG)))
    last_seq = pooling in ('attention', 'mean')
    if n_layers == 1:
        x = RNN(units[0], last_seq)(x)
    else:
        x = RNN(units[0], True)(x)
        if use_bn: x = layers.BatchNormalization()(x)
        x = RNN(units[1], last_seq)(x)
    if pooling == 'attention':  x = TemporalAttention(name='temporal_attention')(x)
    elif pooling == 'mean':     x = layers.GlobalAveragePooling1D()(x)
    return keras.Model(inp, _head(x, n_cls), name=f'proposed_{pooling}')


# ------------------------------------------------------- BASELINES --------
def build_mlp(dim, n_cls):
    inp = _in(dim); x = layers.Flatten()(inp)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    return keras.Model(inp, _head(x, n_cls), name='mlp')

def build_tcnn(dim, n_cls):
    inp = _in(dim); x = inp
    for f in (128, 64):
        x = layers.Conv1D(f, 5, padding='same', activation='relu',
                          kernel_regularizer=regularizers.l2(L2_REG))(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    return keras.Model(inp, _head(x, n_cls), name='temporal_cnn')

def build_cnn_lstm(dim, n_cls):
    inp = _in(dim)
    x = layers.Conv1D(128, 5, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.LSTM(128, return_sequences=False, recurrent_dropout=REC_DROPOUT)(x)
    return keras.Model(inp, _head(x, n_cls), name='cnn_lstm')

def build_uni_lstm(dim, n_cls):
    inp = _in(dim); x = layers.Masking(0.0)(inp)
    x = layers.LSTM(128, return_sequences=True, recurrent_dropout=REC_DROPOUT)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LSTM(64, recurrent_dropout=REC_DROPOUT)(x)
    return keras.Model(inp, _head(x, n_cls), name='uni_lstm')

def build_bigru(dim, n_cls):
    inp = _in(dim); x = layers.Masking(0.0)(inp)
    x = layers.Bidirectional(layers.GRU(128, return_sequences=True,
                                        recurrent_dropout=REC_DROPOUT))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Bidirectional(layers.GRU(64))(x)
    return keras.Model(inp, _head(x, n_cls), name='bigru')

def build_bilstm_plain(dim, n_cls):
    return build_proposed(dim, n_cls, pooling='last')

def build_transformer(dim, n_cls, d_model=128, heads=4, blocks=2):
    inp = _in(dim)
    x = layers.Dense(d_model)(inp)
    pe = layers.Embedding(SEQ_LEN, d_model)(
             layers.Lambda(lambda t: tf.range(tf.shape(t)[1]),
                           output_shape=(SEQ_LEN,))(x))
    x = layers.Add()([x, pe])
    for _ in range(blocks):
        a = layers.MultiHeadAttention(heads, d_model//heads)(x, x)
        x = layers.LayerNormalization()(x + layers.Dropout(0.1)(a))
        f = layers.Dense(d_model*2, activation='relu')(x)
        f = layers.Dense(d_model)(f)
        x = layers.LayerNormalization()(x + layers.Dropout(0.1)(f))
    x = layers.GlobalAveragePooling1D()(x)
    return keras.Model(inp, _head(x, n_cls), name='transformer')

def build_stgcn_lite(dim, n_cls):
    # Lightweight skeleton graph model: treat each landmark as a node, apply a shared
    # spatial projection then temporal convolution. Works on the pose+hands view.
    inp = _in(dim)
    n_nodes = dim // 3 if dim % 3 == 0 else dim // 4
    x = layers.Dense(128, activation='relu')(inp)          # spatial mixing
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 9, padding='same', activation='relu')(x)   # temporal
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 9, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    return keras.Model(inp, _head(x, n_cls), name='stgcn_lite')


BASELINES = {
    'MLP (flattened landmarks)':        build_mlp,
    'Temporal CNN':                     build_tcnn,
    'CNN-LSTM hybrid':                  build_cnn_lstm,
    'Unidirectional LSTM':              build_uni_lstm,
    'BiGRU':                            build_bigru,
    'BiLSTM (no attention)':            build_bilstm_plain,
    'Transformer encoder':              build_transformer,
    'ST-GCN lite (skeleton)':           build_stgcn_lite,
}
print('Baselines registered:', list(BASELINES))
m = build_proposed(DATA['dim'], len(CLASS_LIST))
m.summary()


## 10. Training harness

One protocol for every model: same split, same augmented training set, same optimizer,
callbacks and class-weighting policy, three seeds each.

**Reviewer B comment 11** is answered mechanically here: the class weights are computed from
the actual training counts, and if the corpus turns out to be balanced the cell says so and
disables weighting, which removes the contradiction the reviewer spotted.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

counts = Counter(DATA['ytr'])
imbalance = max(counts.values()) / min(counts.values())
print('Training counts per class:', {CLASS_LIST[k]: v for k, v in sorted(counts.items())})
print(f'Imbalance ratio (max/min): {imbalance:.3f}')

if imbalance < 1.05:
    CLASS_WEIGHT = None
    print('\n-> Corpus is balanced. Class weighting DISABLED.')
    print('   State this in the manuscript: weighting is unnecessary because the balanced')
    print('   factorial design survives the stratified split (answers Reviewer B comment 11).')
else:
    cw = compute_class_weight('balanced', classes=np.unique(DATA['ytr']), y=DATA['ytr'])
    CLASS_WEIGHT = dict(enumerate(cw))
    print('\n-> Residual imbalance detected. Class weights:', 
          {CLASS_LIST[k]: round(v, 3) for k, v in CLASS_WEIGHT.items()})


def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

def train_once(build_fn, data, seed, tag, epochs=EPOCHS, verbose=0):
    set_seed(seed)
    tf.keras.backend.clear_session()
    model = build_fn(data['dim'], len(CLASS_LIST))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR, clipnorm=CLIPNORM),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    ckpt = f'/content/ckpt_{tag}_{seed}.keras'
    cbs = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=ES_PATIENCE,
                                         restore_best_weights=True, verbose=0),
           keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                             patience=RLR_PATIENCE, min_lr=1e-6, verbose=0),
           keras.callbacks.ModelCheckpoint(ckpt, monitor='val_accuracy',
                                           save_best_only=True, verbose=0)]
    t0 = time.time()
    hist = model.fit(data['Xtr'], data['ytr'],
                     validation_data=(data['Xva'], data['yva']),
                     epochs=epochs, batch_size=BATCH_SIZE, class_weight=CLASS_WEIGHT,
                     callbacks=cbs, verbose=verbose)
    train_time = time.time() - t0

    proba = model.predict(data['Xte'], verbose=0)
    pred  = proba.argmax(1)
    acc   = accuracy_score(data['yte'], pred)
    f1m   = f1_score(data['yte'], pred, average='macro')
    return {'model': model, 'history': hist.history, 'pred': pred, 'proba': proba,
            'acc': acc, 'macro_f1': f1m, 'epochs_run': len(hist.history['loss']),
            'train_time_s': train_time, 'params': model.count_params()}

def run_multi_seed(build_fn, data, tag, seeds=SEEDS):
    runs = []
    for s in seeds:
        r = train_once(build_fn, data, s, tag)
        runs.append(r)
        print(f'    seed {s}: acc={r["acc"]*100:.2f}%  macroF1={r["macro_f1"]*100:.2f}%  '
              f'({r["epochs_run"]} epochs, {r["train_time_s"]:.0f}s)')
    accs = np.array([r['acc'] for r in runs]); f1s = np.array([r['macro_f1'] for r in runs])
    return {'runs': runs, 'acc_mean': accs.mean(), 'acc_std': accs.std(),
            'f1_mean': f1s.mean(), 'f1_std': f1s.std(),
            'params': runs[0]['params'],
            'best': runs[int(accs.argmax())]}
print('\nHarness ready.')


## 11. Train the proposed model and every baseline

This is the long cell. With 8 classes and a few thousand augmented training sequences,
expect roughly 10-40 minutes on a T4 GPU for all nine models times three seeds.

In [ ]:
RESULTS = {}

print('=== Proposed: BiLSTM + temporal attention ===')
RESULTS['BiLSTM + temporal attention (proposed)'] = run_multi_seed(
    lambda d, n: build_proposed(d, n), DATA, 'proposed')

for name, fn in BASELINES.items():
    print(f'\n=== Baseline: {name} ===')
    RESULTS[name] = run_multi_seed(fn, DATA, name.replace(' ', '_')[:20])

rows = []
for name, r in RESULTS.items():
    rows.append({'Model': name,
                 'Test accuracy (%)': f"{r['acc_mean']*100:.2f} +/- {r['acc_std']*100:.2f}",
                 'Macro F1 (%)':      f"{r['f1_mean']*100:.2f} +/- {r['f1_std']*100:.2f}",
                 'Parameters':        f"{r['params']:,}",
                 '_sort':             r['acc_mean']})
baseline_tbl = (pd.DataFrame(rows).sort_values('_sort', ascending=False)
                  .drop(columns='_sort').reset_index(drop=True))
print('\n\nBASELINE COMPARISON UNDER AN IDENTICAL PROTOCOL (Reviewer A comment 7)')
display(baseline_tbl)
baseline_tbl.to_csv(f'{OUT_DIR}/table_baseline_comparison.csv', index=False)

# ---- figure with error bars ----
names = [r['Model'] for _, r in baseline_tbl.iterrows()][::-1]
means = [RESULTS[n]['acc_mean']*100 for n in names]
stds  = [RESULTS[n]['acc_std']*100 for n in names]
cols  = ['#2E7D32' if 'proposed' in n else '#64B5F6' for n in names]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(names, means, xerr=stds, color=cols, capsize=4, edgecolor='white')
ax.set_xlabel('Test accuracy (%), mean +/- std over seeds')
ax.set_xlim(0, 105); ax.grid(axis='x', alpha=0.3)
for i, (m_, s_) in enumerate(zip(means, stds)):
    ax.text(m_+s_+1, i, f'{m_:.2f}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_baseline_comparison.png'); plt.show()


## 12. Evaluation of the proposed model — **Reviewer B comment 13**

The confusion matrix is reported as absolute counts with the percentage in parentheses
(`17 (94.4%)`), which is exactly the format the reviewer requested, since with so few test
samples per class a single error moves the percentage by several points.

A McNemar test compares the proposed model against the strongest baseline on the same test
clips, so the claim of superiority rests on a paired statistical test rather than on a gap
between two averages (**Reviewer A comment 9**).

In [ ]:
best = RESULTS['BiLSTM + temporal attention (proposed)']['best']
yte, pred = DATA['yte'], best['pred']

print(classification_report(yte, pred, target_names=CLASS_LIST, digits=4))

prec, rec, f1, sup = precision_recall_fscore_support(yte, pred, labels=range(len(CLASS_LIST)))
per_class_tbl = pd.DataFrame({
    'Class': CLASS_LIST,
    'Precision (%)': (prec*100).round(2), 'Recall (%)': (rec*100).round(2),
    'F1 (%)': (f1*100).round(2), 'Support': sup})
per_class_tbl.loc[len(per_class_tbl)] = ['Macro average', round(prec.mean()*100, 2),
                                         round(rec.mean()*100, 2), round(f1.mean()*100, 2),
                                         sup.sum()]
display(per_class_tbl)
per_class_tbl.to_csv(f'{OUT_DIR}/table_per_class_metrics.csv', index=False)

# ---- confusion matrix: counts AND percentages ----
cm = confusion_matrix(yte, pred, labels=range(len(CLASS_LIST)))
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
annot = np.array([[f'{cm[i,j]}\n({cm_pct[i,j]:.1f}%)' if cm[i,j] else '0'
                   for j in range(len(CLASS_LIST))] for i in range(len(CLASS_LIST))])

fig, ax = plt.subplots(figsize=(1.15*len(CLASS_LIST)+2, 1.0*len(CLASS_LIST)+1.5))
im = ax.imshow(cm, cmap='Blues')
for i in range(len(CLASS_LIST)):
    for j in range(len(CLASS_LIST)):
        ax.text(j, i, annot[i, j], ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > cm.max()*0.6 else 'black')
ax.set_xticks(range(len(CLASS_LIST))); ax.set_xticklabels(CLASS_LIST, rotation=35, ha='right')
ax.set_yticks(range(len(CLASS_LIST))); ax.set_yticklabels(CLASS_LIST)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix: count (percentage of true class)')
plt.colorbar(im, shrink=0.8); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_confusion_matrix_counts.png'); plt.show()

cm_df = pd.DataFrame(annot, index=CLASS_LIST, columns=CLASS_LIST)
cm_df.to_csv(f'{OUT_DIR}/table_confusion_matrix_counts.csv')

# ---- McNemar against the strongest baseline ----
from statsmodels.stats.contingency_tables import mcnemar
bl_names = [n for n in RESULTS if 'proposed' not in n]
strongest = max(bl_names, key=lambda n: RESULTS[n]['acc_mean'])
bl_pred = RESULTS[strongest]['best']['pred']

a = (pred == yte); b = (bl_pred == yte)
table = [[int(( a &  b).sum()), int(( a & ~b).sum())],
         [int((~a &  b).sum()), int((~a & ~b).sum())]]
res = mcnemar(table, exact=True)
print(f'\nMcNemar, proposed vs strongest baseline ({strongest}):')
print(f'  both correct={table[0][0]}, proposed only={table[0][1]}, '
      f'baseline only={table[1][0]}, both wrong={table[1][1]}')
print(f'  p-value = {res.pvalue:.4f}  ->', 
      'difference is statistically significant (p<0.05)' if res.pvalue < 0.05
      else 'difference is NOT statistically significant at p<0.05')
json.dump({'comparator': strongest, 'table': table, 'p_value': float(res.pvalue)},
          open(f'{OUT_DIR}/mcnemar_test.json', 'w'), indent=2)


## 13. Ablation study — A4, A5, **A10**, **B16**

Ten variants, each differing from the full model in exactly one decision. The `with_face`
variant is the important addition: it tests empirically whether adding the 468 face
landmarks helps, which is what turns the face-exclusion argument in **Reviewer A comment 10**
from an assertion into a measurement.

In [ ]:
ABL = {}

def ablate(name, build_fn, data=None, aug=True):
    d = data or DATA
    if not aug:
        norm = Normalizer().fit(X_raw[tr_i][:, :, SLICES['full_258']])
        d = {'Xtr': norm.transform(X_raw[tr_i][:, :, SLICES['full_258']]),
             'ytr': y_idx[tr_i],
             'Xva': norm.transform(X_raw[va_i][:, :, SLICES['full_258']]), 'yva': y_idx[va_i],
             'Xte': norm.transform(X_raw[te_i][:, :, SLICES['full_258']]), 'yte': y_idx[te_i],
             'dim': FEAT_DIM}
    print(f'--- {name}')
    ABL[name] = run_multi_seed(build_fn, d, name.replace(' ', '_')[:18], seeds=SEEDS[:2])

ablate('Full model (reference)',        lambda d, n: build_proposed(d, n))
ablate('Attention -> mean pooling',     lambda d, n: build_proposed(d, n, pooling='mean'))
ablate('Attention -> last hidden state',lambda d, n: build_proposed(d, n, pooling='last'))
ablate('Unidirectional (no backward)',  lambda d, n: build_proposed(d, n, bidirectional=False))
ablate('Single BiLSTM layer',           lambda d, n: build_proposed(d, n, n_layers=1))
ablate('No batch normalization',        lambda d, n: build_proposed(d, n, use_bn=False))
ablate('Hands-only features',           lambda d, n: build_proposed(d, n),
       data=build_view('hands_only'))
ablate('Pose-only features',            lambda d, n: build_proposed(d, n),
       data=build_view('pose_only'))
if INCLUDE_FACE:
    ablate('With face landmarks added',  lambda d, n: build_proposed(d, n),
           data=build_view('with_face'))
ablate('No data augmentation',          lambda d, n: build_proposed(d, n), aug=False)

ref = ABL['Full model (reference)']['acc_mean']
abl_rows = []
for k, v in ABL.items():
    abl_rows.append({'Variant': k,
                     'Test accuracy (%)': round(v['acc_mean']*100, 2),
                     'Std (%)': round(v['acc_std']*100, 2),
                     'Delta vs full (pp)': round((v['acc_mean']-ref)*100, 2)})
abl_tbl = pd.DataFrame(abl_rows).sort_values('Delta vs full (pp)', ascending=False)
print('\nABLATION RESULTS')
display(abl_tbl)
abl_tbl.to_csv(f'{OUT_DIR}/table_ablation.csv', index=False)

face_row = abl_tbl[abl_tbl.Variant == 'With face landmarks added']
if len(face_row):
    d_face = face_row['Delta vs full (pp)'].values[0]
    print(f'\nFACE-LANDMARK FINDING (Reviewer A comment 10): adding the face stream changes '
          f'accuracy by {d_face:+.2f} pp.')
    print('  Report this measured value in Section 3.2.6 instead of arguing the exclusion '
          'on efficiency grounds alone.')


## 14. Attention weights — **Reviewer A comment 5**

Extracts the actual `alpha` distribution over the 60 frames for test clips and plots it.
This is the empirical backing for the saliency claim in Section 4.6.

In [ ]:
model = best['model']
att_layer = model.get_layer('temporal_attention')
# the tensor feeding the attention layer is its own input; with masking enabled
# Keras may hand back [tensor, mask], so take the first element
att_input = att_layer.input
if isinstance(att_input, (list, tuple)):
    att_input = att_input[0]
feat_extractor = keras.Model(model.inputs, att_input)

H = feat_extractor.predict(DATA['Xte'], verbose=0)
if isinstance(H, (list, tuple)):
    H = H[0]
H = np.asarray(H)
print('Sequence fed to attention:', H.shape, '(clips, frames, 2 x 64 units)')
e = np.tanh(H @ att_layer.W.numpy() + att_layer.b.numpy()) @ att_layer.v.numpy()
e = e.squeeze(-1)
alpha = tf.nn.softmax(e, axis=1).numpy()
np.save(f'{OUT_DIR}/attention_weights_test.npy', alpha)

n_show = min(6, len(CLASS_LIST))
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, ci in zip(axes.flatten(), range(n_show)):
    idxs = np.where(DATA['yte'] == ci)[0]
    if not len(idxs):
        ax.axis('off'); continue
    a = alpha[idxs[0]]
    ax.fill_between(range(1, SEQ_LEN+1), a, alpha=0.55, color='#1565C0')
    ax.plot(range(1, SEQ_LEN+1), a, color='#1565C0', lw=1.2)
    ax.axvline(a.argmax()+1, color='#E65100', ls='--', lw=1.6)
    ax.set_title(CLASS_LIST[ci], fontsize=10, fontweight='bold')
    ax.set_xlabel('Frame'); ax.set_ylabel('Attention weight'); ax.grid(alpha=0.2)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_attention_weights.png'); plt.show()

peak = pd.DataFrame({'Class': [CLASS_LIST[c] for c in DATA['yte']],
                     'Peak frame': alpha.argmax(1)+1,
                     'Peak weight': alpha.max(1),
                     'Entropy': -(alpha*np.log(alpha+1e-12)).sum(1)})
summary = peak.groupby('Class').agg(['mean', 'std']).round(3)
print('Attention concentration per class (low entropy = more selective):')
display(summary)
summary.to_csv(f'{OUT_DIR}/table_attention_stats.csv')


## 15. Gesture characterization — **Reviewer B comment 4**

The reviewer asks for the linguistic and visual characteristics of the selected gestures and
for them to be correlated with the ablation. Rather than asserting phonological descriptions,
this cell measures them from the corpus: handedness, articulation height relative to the
shoulder line, and movement magnitude. Those are observable facts you can state in
Section 3.2.6 and tie directly to the hands-only / pose-only ablation result.

In [ ]:
L_SH, R_SH = 11, 12
char_rows = []
for ci, cls in enumerate(CLASS_LIST):
    idxs = np.where(y_idx == ci)[0]
    lh_use, rh_use, both, heights, motion = [], [], [], [], []
    for i in idxs:
        seq = X_raw[i]
        live = ~np.all(seq == 0, axis=1)
        if live.sum() == 0: continue
        s = seq[live]
        lh = s[:, POSE_DIM:POSE_DIM+HAND_DIM]
        rh = s[:, POSE_DIM+HAND_DIM:FEAT_DIM]
        lh_on = ~np.all(lh == 0, axis=1); rh_on = ~np.all(rh == 0, axis=1)
        lh_use.append(lh_on.mean()); rh_use.append(rh_on.mean())
        both.append((lh_on & rh_on).mean())
        pose = s[:, :POSE_DIM].reshape(len(s), 33, 4)
        sh_y = (pose[:, L_SH, 1] + pose[:, R_SH, 1]) / 2
        wrist_y = []
        if rh_on.any(): wrist_y.append(rh[rh_on].reshape(-1, 21, 3)[:, 0, 1].mean())
        if lh_on.any(): wrist_y.append(lh[lh_on].reshape(-1, 21, 3)[:, 0, 1].mean())
        if wrist_y: heights.append(sh_y.mean() - np.mean(wrist_y))  # >0 = above shoulders
        hands = s[:, POSE_DIM:FEAT_DIM]
        motion.append(np.abs(np.diff(hands, axis=0)).mean() if len(hands) > 1 else 0)
    char_rows.append({
        'Class': cls, 'Clips': len(idxs),
        'Left hand present (%)':  round(np.mean(lh_use)*100, 1),
        'Right hand present (%)': round(np.mean(rh_use)*100, 1),
        'Two-handed frames (%)':  round(np.mean(both)*100, 1),
        'Handedness': 'two-handed' if np.mean(both) > 0.5 else 'one-handed',
        'Hand height vs shoulders': round(float(np.mean(heights)), 4) if heights else np.nan,
        'Mean frame-to-frame motion': round(float(np.mean(motion)), 5)})

char_tbl = pd.DataFrame(char_rows)
print('MEASURED VISUAL CHARACTERISTICS PER GESTURE (evidence for Section 3.2.6)')
display(char_tbl)
char_tbl.to_csv(f'{OUT_DIR}/table_gesture_characteristics.csv', index=False)
print('\nPositive "hand height vs shoulders" means the hands sit above the shoulder line.')
print('Use these with the hands-only / pose-only ablation: classes articulated away from')
print('neutral space are the ones that lose most when the pose stream is removed.')


## 16. Signer-independent evaluation — **Reviewer B comment 15**

Leave-one-signer-out. If you have two signers this gives two folds and the estimate is
weak, but running it is still far better than declaring it impossible: it yields a real
number for the manuscript and quantifies the gap between signer-dependent and
signer-independent performance.

In [ ]:
signers = sorted(set(g_raw))
print('Signers available:', signers)

loso_rows = []
if len(signers) < 2:
    print('Only one signer present; leave-one-signer-out is not applicable.')
else:
    for held in signers:
        te_mask = (g_raw == held)
        tr_pool = np.where(~te_mask)[0]
        te_idx  = np.where(te_mask)[0]
        if len(tr_pool) < 20 or len(te_idx) < len(CLASS_LIST):
            print(f'Skipping {held}: not enough data.'); continue
        tr2, va2 = train_test_split(tr_pool, test_size=0.15,
                                    stratify=y_idx[tr_pool], random_state=SPLIT_SEED)
        Xa, ya, _, _ = augment_training_set(X_raw[tr2], y_idx[tr2], g_raw[tr2])
        norm = Normalizer().fit(Xa[:, :, SLICES['full_258']])
        d = {'Xtr': norm.transform(Xa[:, :, SLICES['full_258']]), 'ytr': ya,
             'Xva': norm.transform(X_raw[va2][:, :, SLICES['full_258']]), 'yva': y_idx[va2],
             'Xte': norm.transform(X_raw[te_idx][:, :, SLICES['full_258']]), 'yte': y_idx[te_idx],
             'dim': FEAT_DIM}
        print(f'\n--- Held-out signer: {held}  (train {len(tr2)}, test {len(te_idx)})')
        r = train_once(lambda dim, n: build_proposed(dim, n), d, SEEDS[0], f'loso_{held}')
        print(f'    accuracy = {r["acc"]*100:.2f}%   macro F1 = {r["macro_f1"]*100:.2f}%')
        loso_rows.append({'Held-out signer': held, 'Train clips': len(tr2),
                          'Test clips': len(te_idx),
                          'Accuracy (%)': round(r['acc']*100, 2),
                          'Macro F1 (%)': round(r['macro_f1']*100, 2)})

if loso_rows:
    loso = pd.DataFrame(loso_rows)
    loso.loc[len(loso)] = ['Mean', '', '', round(loso['Accuracy (%)'].mean(), 2),
                           round(loso['Macro F1 (%)'].mean(), 2)]
    print('\nSIGNER-INDEPENDENT (leave-one-signer-out) RESULTS')
    display(loso)
    loso.to_csv(f'{OUT_DIR}/table_signer_independent.csv', index=False)
    dep = RESULTS['BiLSTM + temporal attention (proposed)']['acc_mean']*100
    ind = loso['Accuracy (%)'].iloc[-1]
    print(f'\nSigner-dependent : {dep:.2f}%')
    print(f'Signer-independent: {ind:.2f}%   (gap {dep-ind:+.2f} pp)')
    print('Report BOTH in the manuscript. The gap is the honest measure of how much the')
    print('headline figure depends on having seen the signer during training.')


## 17. Parameter audit — **Reviewer B comment 10**

The reviewer is right that the published layer table implies far more parameters than the
177,607 quoted, and that "weight sharing" was never properly described. This cell prints the
true layer-by-layer counts so the manuscript can simply state the real number.

In [ ]:
pm = RESULTS['BiLSTM + temporal attention (proposed)']['best']['model']
rows = []
def _oshape(l):
    try:
        return str(tuple(l.output.shape))
    except Exception:
        try:
            return str(l.output_shape)
        except Exception:
            return 'n/a'

for l in pm.layers:
    rows.append({'Layer': l.name, 'Type': l.__class__.__name__,
                 'Output shape': _oshape(l),
                 'Parameters': l.count_params()})
param_tbl = pd.DataFrame(rows)
param_tbl.loc[len(param_tbl)] = ['TOTAL', '', '', pm.count_params()]
display(param_tbl)
param_tbl.to_csv(f'{OUT_DIR}/table_parameter_audit.csv', index=False)

total = pm.count_params()
print(f'\nTRUE total trainable parameters: {total:,}')
print(f'Manuscript currently claims      : 177,607')
if abs(total - 177607) > 1000:
    print('\n-> These do not match. Replace the figure in the abstract, Table 5, Table 11')
    print('   and Section 4.7.2 with the value above, and delete the "weight sharing"')
    print('   footnote, which does not describe anything in this architecture.')

comp = pd.DataFrame([{'Model': n, 'Parameters': f"{r['params']:,}",
                      'Accuracy (%)': round(r['acc_mean']*100, 2)}
                     for n, r in RESULTS.items()]).sort_values('Parameters')
display(comp)
comp.to_csv(f'{OUT_DIR}/table_params_vs_accuracy.csv', index=False)


## 18. Runtime benchmark — **Reviewer B comment 14**

Measures landmark extraction throughput, model inference latency and end-to-end delay, so
that any claim about interactive use rests on a measurement. Run this on the hardware you
intend to describe; note in the manuscript which device produced the numbers.

In [ ]:
import platform, subprocess

# ---- landmark extraction throughput ----
sample = kept.sample(min(5, len(kept)), random_state=0)
ext_times, n_frames_total = [], 0
with mp_holistic.Holistic(static_image_mode=False, model_complexity=1,
                          min_detection_confidence=MIN_DET_CONF,
                          min_tracking_confidence=MIN_TRK_CONF) as h:
    for _, r in sample.iterrows():
        path = inv[inv.filename == r['filename']]['path'].values[0]
        cap = cv2.VideoCapture(path); frames = []
        while True:
            ok, fr = cap.read()
            if not ok: break
            frames.append(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        cap.release()
        t0 = time.time()
        for fr in frames: h.process(fr)
        ext_times.append(time.time()-t0); n_frames_total += len(frames)

ext_fps = n_frames_total / sum(ext_times)
ext_ms  = 1000/ext_fps

# ---- model inference latency (batch size 1) ----
x1 = DATA['Xte'][:1]
for _ in range(10): pm.predict(x1, verbose=0)          # warm-up
t0 = time.time()
for _ in range(50): pm.predict(x1, verbose=0)
inf_ms = (time.time()-t0)/50*1000

bench = pd.DataFrame([
    {'Stage': 'MediaPipe Holistic landmark extraction',
     'Measurement': f'{ext_ms:.1f} ms/frame ({ext_fps:.1f} fps)'},
    {'Stage': 'Recognition model inference (60-frame clip, batch 1)',
     'Measurement': f'{inf_ms:.1f} ms'},
    {'Stage': 'End-to-end for one 60-frame clip (extraction + inference)',
     'Measurement': f'{ext_ms*SEQ_LEN + inf_ms:.0f} ms'},
    {'Stage': 'Real-time headroom at 30 fps (33.3 ms budget/frame)',
     'Measurement': 'within budget' if ext_ms < 33.3 else f'EXCEEDS budget by {ext_ms-33.3:.1f} ms/frame'},
    {'Stage': 'Device', 'Measurement': ('GPU: ' + tf.config.list_physical_devices('GPU')[0].name)
                                       if tf.config.list_physical_devices('GPU') else 'CPU only'},
    {'Stage': 'Platform', 'Measurement': platform.platform()},
])
display(bench)
bench.to_csv(f'{OUT_DIR}/table_runtime_benchmark.csv', index=False)
print('\nQuote these numbers in Section 4.7 and state the device. Only claim real-time')
print('operation if extraction stays inside the 33.3 ms per-frame budget on the TARGET')
print('device, not on a Colab GPU.')


## 19. Speech synthesis stage — **Reviewer B comment 7**

Documents the checkpoint, whether fine-tuning was applied, the inference configuration and
how audio is produced, and generates one audio file per class.

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer
from scipy.io.wavfile import write as wav_write
from IPython.display import Audio, display as ipy_display

CHECKPOINT = 'facebook/mms-tts-swh'
tts_tok   = AutoTokenizer.from_pretrained(CHECKPOINT)
tts_model = VitsModel.from_pretrained(CHECKPOINT)
tts_model.eval()

SR = tts_model.config.sampling_rate
tts_cfg = {
    'checkpoint': CHECKPOINT, 'architecture': 'VITS (MMS)',
    'fine_tuned': False, 'speaker_conditioning': 'single speaker, none available',
    'sampling_rate_hz': SR,
    'noise_scale': float(getattr(tts_model.config, 'noise_scale', float('nan'))),
    'noise_scale_duration': float(getattr(tts_model.config, 'noise_scale_duration', float('nan'))),
    'speaking_rate': float(getattr(tts_model.config, 'speaking_rate', float('nan'))),
    'vocab_size': int(tts_model.config.vocab_size),
    'parameters': sum(p.numel() for p in tts_model.parameters()),
}
print(json.dumps(tts_cfg, indent=2))
json.dump(tts_cfg, open(f'{OUT_DIR}/tts_config.json', 'w'), indent=2)

# Label -> Swahili text lookup. EDIT the right-hand side to the wording you want spoken.
LABEL_TO_TEXT = {c: c for c in CLASS_LIST}
LABEL_TO_TEXT.update({'habari': 'habari', 'baba': 'baba', 'mama': 'mama',
                      'kula': 'kula', 'nenda': 'nenda', 'njema': 'njema',
                      'hedhi': 'hedhi', 'siku': 'siku'})
print('\nLabel to text mapping:', LABEL_TO_TEXT)
json.dump(LABEL_TO_TEXT, open(f'{OUT_DIR}/label_to_text.json', 'w'), indent=2)

def normalize_text(s):
    return ''.join(ch for ch in s.lower().strip() if ch.isalpha() or ch.isspace())

os.makedirs(f'{OUT_DIR}/audio', exist_ok=True)
synth_ms = []
for cls in CLASS_LIST:
    text = normalize_text(LABEL_TO_TEXT[cls])
    inputs = tts_tok(text, return_tensors='pt')
    t0 = time.time()
    with torch.no_grad():
        wave = tts_model(**inputs).waveform
    synth_ms.append((time.time()-t0)*1000)
    audio = wave.squeeze().cpu().numpy()
    wav_write(f'{OUT_DIR}/audio/{cls}.wav', SR, (audio*32767).astype(np.int16))
    print(f'  {cls:10s} -> "{text}"  {len(audio)/SR:.2f}s')

print(f'\nMean synthesis latency: {np.mean(synth_ms):.0f} ms per token')
ipy_display(Audio(f'{OUT_DIR}/audio/{CLASS_LIST[0]}.wav'))

# ---- full pipeline demonstration on one test clip ----
i = 0
p = pm.predict(DATA['Xte'][i:i+1], verbose=0)[0]
pred_cls = CLASS_LIST[p.argmax()]
print(f'\nEnd-to-end demo: true={CLASS_LIST[DATA["yte"][i]]}  '
      f'predicted={pred_cls}  confidence={p.max():.3f}')
ipy_display(Audio(f'{OUT_DIR}/audio/{pred_cls}.wav'))


## 20. Diagrams — **Reviewer A comment 15** and **Reviewer B comment 5**

Two figures: the landmark pipeline from frame to tensor, and the system architecture with
the recognition stage visually separated from the downstream synthesis stage.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def box(ax, x, y, w, h, text, fc, tc='white', fs=9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02',
                                facecolor=fc, edgecolor='white', lw=1.5, zorder=2))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', color=tc,
            fontsize=fs, fontweight='bold', zorder=3, linespacing=1.4)

def arrow(ax, x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                                 mutation_scale=14, color='#546E7A', lw=1.8, zorder=1))

# ---- Figure: landmark extraction pipeline (B5) ----
fig, ax = plt.subplots(figsize=(14, 3.4)); ax.set_xlim(0, 15); ax.set_ylim(0, 3.4); ax.axis('off')
stages = [(0.2,'Video clip\n30 fps','#0D47A1'), (2.6,'MediaPipe\nHolistic','#1565C0'),
          (5.0,'33 pose x 4 = 132\n21 LH x 3 = 63\n21 RH x 3 = 63','#1976D2'),
          (7.6,f'258-d vector\nper frame','#42A5F5'),
          (10.0,f'Normalize\n({NORMALIZATION})','#66BB6A'),
          (12.4,f'{SEQ_LEN} x 258\ntensor','#2E7D32')]
for x, t, c in stages: box(ax, x, 1.0, 2.1, 1.5, t, c)
for i in range(len(stages)-1): arrow(ax, stages[i][0]+2.15, 1.75, stages[i+1][0]-0.05, 1.75)
ax.text(7.5, 3.05, 'Landmark extraction pipeline', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_landmark_pipeline.png'); plt.show()

# ---- Figure: system architecture, recognition vs synthesis (A15) ----
fig, ax = plt.subplots(figsize=(14, 5)); ax.set_xlim(0, 15); ax.set_ylim(0, 5); ax.axis('off')
ax.add_patch(FancyBboxPatch((0.1, 1.1), 10.3, 2.7, boxstyle='round,pad=0.05',
                            facecolor='#E3F2FD', edgecolor='#1565C0', lw=2, ls='--', zorder=0))
ax.text(5.25, 3.95, 'Recognition stage — contribution of this study',
        ha='center', fontsize=11, fontweight='bold', color='#0D47A1')
ax.add_patch(FancyBboxPatch((10.7, 1.1), 4.1, 2.7, boxstyle='round,pad=0.05',
                            facecolor='#FFF3E0', edgecolor='#E65100', lw=2, ls='--', zorder=0))
ax.text(12.75, 3.95, 'Synthesis stage — pretrained,\nused as released',
        ha='center', fontsize=11, fontweight='bold', color='#BF360C')
rec = [(0.4,'Video\ninput','#0D47A1'), (2.4,'MediaPipe\nHolistic','#1565C0'),
       (4.4,f'{SEQ_LEN} x 258\nsequence','#1976D2'),
       (6.4,'BiLSTM\n128 + 64','#2E7D32'), (8.4,'Temporal\nattention','#388E3C')]
for x, t, c in rec: box(ax, x, 1.7, 1.8, 1.5, t, c)
for i in range(len(rec)-1): arrow(ax, rec[i][0]+1.85, 2.45, rec[i+1][0]-0.05, 2.45)
box(ax, 11.0, 1.7, 1.6, 1.5, 'Label to\nSwahili text', '#F57F17')
box(ax, 13.0, 1.7, 1.6, 1.5, 'MMS-VITS\nspeech', '#C62828')
arrow(ax, 10.25, 2.45, 10.95, 2.45); arrow(ax, 12.65, 2.45, 12.95, 2.45)
ax.text(10.55, 1.35, 'gesture\nlabel', ha='center', fontsize=7.5, style='italic', color='#37474F')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_system_architecture.png'); plt.show()
print('Saved both diagrams. Figure 1 in the manuscript should be replaced with the second one.')


## 21. Export everything

In [ ]:
manifest = sorted(os.listdir(OUT_DIR))
print('Files in outputs/:')
for f in manifest:
    print('  ', f)

# A single summary sheet mapping every reviewer comment to its evidence file
mapping = pd.DataFrame([
 ['A6',  'Split before augmentation; exact counts', 'table_split_and_augmentation.csv'],
 ['A7',  'Baselines under identical protocol',      'table_baseline_comparison.csv'],
 ['A9',  'Paired significance test',                'mcnemar_test.json'],
 ['A10', 'Face landmarks tested empirically',       'table_ablation.csv (with_face row)'],
 ['A15', 'System architecture diagram',             'fig_system_architecture.png'],
 ['B4',  'Measured gesture characteristics',        'table_gesture_characteristics.csv'],
 ['B5',  'Landmark pipeline diagram',               'fig_landmark_pipeline.png'],
 ['B6',  'Normalization scope, leak-free',          'config.json (normalization field)'],
 ['B7',  'TTS checkpoint and inference config',     'tts_config.json, label_to_text.json'],
 ['B10', 'True parameter counts',                   'table_parameter_audit.csv'],
 ['B11', 'Class-weighting decision from data',      'printed in Section 10 + config.json'],
 ['B13', 'Confusion matrix as counts (pct)',        'table_confusion_matrix_counts.csv'],
 ['B14', 'Runtime benchmark',                       'table_runtime_benchmark.csv'],
 ['B15', 'Signer-independent evaluation',           'table_signer_independent.csv'],
 ['B16', 'Feature ablation, empirical framing',     'table_ablation.csv'],
], columns=['Reviewer comment', 'Evidence produced', 'File'])
mapping.to_csv(f'{OUT_DIR}/REVIEWER_EVIDENCE_MAP.csv', index=False)
display(mapping)

shutil.make_archive('/content/swsl_results', 'zip', OUT_DIR)
from google.colab import files
files.download('/content/swsl_results.zip')
print('\nDownload started. Send this zip back and the manuscript numbers can be updated '
      'directly from it.')
